In [1]:
import pandas as pd
import numpy as np
import pickle, itertools, time
from implicit.nearest_neighbours import CosineRecommender, BM25Recommender
from implicit.als import AlternatingLeastSquares
from recsys_utils import evaluate, implicit_recs, METRIC_COLS

split = pickle.load(open('../data/processed/split.pkl', 'rb'))
users = split['eval_users']
K = 5
train = split['train']
common = dict(item_pop=split['pop_train'], n_users=split['n_users'], n_items=split['n_items'], k=K)

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Այստեղ կատարել եմ KNN մոդելի hyperparameter tuning:Կիրառել եմ grid search մոտեցումը և փորձարկել cosineRecommender-ը և BM25Recommender-ը, որոնցից յուրաքանչյուրի համար վերցրել եմ հարևանների չորս տարբեր քանակ՝ 20, 50, 100 և 200:Որպես գնահատման չափանիշ օգտագործել եմ NDCG@5-ը, որը թույլ է տալիս գնահատել ոչ միայն ճիշտ հայտարարության հայտնվելը ցանկում, այլև դրա դիրքը  առաջին 5 առաջարկությունների մեջ:

In [2]:
knn_rows = []
knn_grid = [(n, cls, nb) for (n, cls) in [('Cosine', CosineRecommender), ('BM25', BM25Recommender)]
                         for nb in [20, 50, 100, 200]]

for name, cls, nb in knn_grid:
    t0 = time.time()
    model = cls(K=nb)
    model.fit(train)
    recs = implicit_recs(model, users, train, k=K)
    r = evaluate(recs, split['val_truth'], **common)
    knn_rows.append({'similarity': name, 'neighbors': nb, **r.to_dict(), 'sec': round(time.time() - t0, 1)})
    print(f"{name:6s} K={nb:<4d} NDCG@5={r['NDCG@5']:.5f}")

knn_tuning = pd.DataFrame(knn_rows).sort_values('NDCG@5', ascending=False)
display(knn_tuning.round(5))
best_knn = knn_tuning.iloc[0]

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0076923370361328125 seconds
  warnings.warn(
  0%|          | 0/279599 [00:00<?, ?it/s]

100%|██████████| 279599/279599 [00:00<00:00, 348579.29it/s]
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.003810405731201172 seconds
  warnings.warn(


Cosine K=20   NDCG@5=0.00284


100%|██████████| 279599/279599 [00:01<00:00, 251625.97it/s]
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.002262115478515625 seconds
  warnings.warn(


Cosine K=50   NDCG@5=0.00292


...

100%|██████████| 279599/279599 [00:00<00:00, 317744.43it/s]
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.002249002456665039 seconds
  warnings.warn(


BM25   K=100  NDCG@5=0.00277


100%|██████████| 279599/279599 [00:00<00:00, 328984.05it/s]


BM25   K=200  NDCG@5=0.00277


,similarity,neighbors,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty,sec
1,Cosine,50,0.00096,0.0048,0.00231,0.00292,0.09464,16.22566,2.4
2,Cosine,100,0.00096,0.0048,0.00231,0.00292,0.09465,16.22515,2.9
3,Cosine,200,0.00096,0.0048,0.00231,0.00292,0.09465,16.22515,5.6
0,Cosine,20,0.00092,0.0046,0.00227,0.00284,0.09454,16.22935,1.7
4,BM25,20,0.00090,0.0045,0.00226,0.00280,0.09707,16.21774,2.8
5,BM25,50,0.00090,0.0045,0.00221,0.00277,0.09709,16.21735,2.5
6,BM25,100,0.00090,0.0045,0.00221,0.00277,0.09709,16.21735,2.8
7,BM25,200,0.00090,0.0045,0.00221,0.00277,0.09709,16.21735,4.0


այստեղ կատարել եմ Grid Search-ով թյունինգ ենք  ALS մատրիցային ֆակտորիզացիան։ Լավագույն արդյունքը ստացել եմ factors=128 և alpha=100 դեպքում (NDCG@5 = 0.0083)․ alpha=100-ը կտրուկ բարձրացրեց ճշգրտությունը, իսկ 128 գործոնը ապահովեց 18 անգամ ավելի արագ աշխատանք, քան 256-ը։

In [3]:
als_rows = []
als_grid = list(itertools.product([128, 256],           
                                  [0.01, 0.1],          
                                  [10.0, 40.0, 100.0],  
                                  [15]))                

for factors, reg, alpha, iters in als_grid:
    t0 = time.time()
    model = AlternatingLeastSquares(factors=factors, regularization=reg, alpha=alpha,
                                    iterations=iters, random_state=42)
    model.fit(train)
    recs = implicit_recs(model, users, train, k=K)
    r = evaluate(recs, split['val_truth'], **common)
    als_rows.append({'factors': factors, 'regularization': reg, 'alpha': alpha,
                     'iterations': iters, **r.to_dict(), 'sec': round(time.time() - t0, 1)})
    print(f"factors={factors:<4d} reg={reg:<5} alpha={alpha:<5} NDCG@5={r['NDCG@5']:.5f}")

als_tuning = pd.DataFrame(als_rows).sort_values('NDCG@5', ascending=False)
display(als_tuning.round(5))
best_als = als_tuning.iloc[0]

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 20 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [00:16<00:00,  1.13s/it]


factors=128  reg=0.01  alpha=10.0  NDCG@5=0.00604


100%|██████████| 15/15 [00:12<00:00,  1.21it/s]


factors=128  reg=0.01  alpha=40.0  NDCG@5=0.00790


100%|██████████| 15/15 [00:13<00:00,  1.09it/s]


...

100%|██████████| 15/15 [09:16<00:00, 37.11s/it]


factors=256  reg=0.1   alpha=40.0  NDCG@5=0.00818


100%|██████████| 15/15 [07:39<00:00, 30.66s/it]


factors=256  reg=0.1   alpha=100.0 NDCG@5=0.00821


,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty,sec
2,128,0.01,100.0,15,0.00244,0.0122,0.00703,0.00830,0.04172,13.29500,22.3
5,128,0.10,100.0,15,0.00244,0.0122,0.00702,0.00830,0.04169,13.28945,22.4
8,256,0.01,100.0,15,0.00248,0.0124,0.00685,0.00822,0.05810,13.68814,403.2
11,256,0.10,100.0,15,0.00246,0.0123,0.00686,0.00821,0.05814,13.67133,470.6
10,256,0.10,40.0,15,0.00246,0.0123,0.00682,0.00818,0.04230,13.35255,569.2
7,256,0.01,40.0,15,0.00244,0.0122,0.00680,0.00814,0.04233,13.36524,394.1
1,128,0.01,40.0,15,0.00232,0.0116,0.00669,0.00790,0.02806,13.01632,20.5
4,128,0.10,40.0,15,0.00232,0.0116,0.00666,0.00788,0.02804,13.01340,25.9
6,256,0.01,10.0,15,0.00234,0.0117,0.00611,0.00748,0.02240,12.97128,383.1
9,256,0.10,10.0,15,0.00232,0.0116,0.00609,0.00745,0.02239,12.96944,394.6


Այստեղ վերցրել եմ լավագույն հայտնաբերված պարամետրերը, սովորեցրել  մոդելները (KNN և ALS) ամբողջական trainval բազայի վրա և գնահատել դրանց արդյունավետությունը թեստային  տվյալների վրա։ Այստեղ պարզ երևում է, որ ALS-ը զգալիորեն գերազանցում է KNN-ին։

In [4]:
trainval = split['trainval']
final_args = dict(item_pop=split['pop_trainval'], n_users=split['n_users'],
                  n_items=split['n_items'], k=K)

knn_cls = CosineRecommender if best_knn['similarity'] == 'Cosine' else BM25Recommender
knn_model = knn_cls(K=int(best_knn['neighbors']))
knn_model.fit(trainval)
res_knn = evaluate(implicit_recs(knn_model, users, trainval, k=K), split['test_truth'], **final_args)

als_model = AlternatingLeastSquares(factors=int(best_als['factors']),
                                    regularization=float(best_als['regularization']),
                                    alpha=float(best_als['alpha']),
                                    iterations=int(best_als['iterations']), random_state=42)
als_model.fit(trainval)
res_als = evaluate(implicit_recs(als_model, users, trainval, k=K), split['test_truth'], **final_args)

model_results = pd.DataFrame({'KNN': res_knn, 'ALS': res_als}).T[METRIC_COLS]
model_results.index.name = 'Model'
model_results

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.005840301513671875 seconds
  warnings.warn(
100%|██████████| 15/15 [00:15<00:00,  1.04s/it]


,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty
Model,,,,,,
KNN,0.00098,0.0049,0.002750,0.003285,0.117762,16.264257
ALS,0.00260,0.0130,0.007267,0.008678,0.033133,12.591618


Այստեղ կառուցել եմ Content-Based մոդել։ Այն օգտատիրոջ պատմությունից ստանում է նրա նախասիրությունների վեկտորը և կոսինուսային նմանությամբ գտնում նմանատիպ հայտարարություններ։ Կարևոր առանձնահատկությունը քաղաքային ֆիլտրն է. համակարգը ավտոմատ որոշում է օգտատիրոջ նախընտրած քաղաքը և արգելափակում մյուս քաղաքների առաջարկները։

In [5]:
item_features = pd.read_pickle('../data/processed/item_features.pkl')
listings_clean = pd.read_pickle('../data/processed/listings_clean.pkl')

X = item_features.astype(np.float32).values
X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

city_codes = listings_clean['city'].astype('category').cat.codes.values.astype(int) + 1

def content_recs(users, user_items_csr, k=5, batch=200):
    recs = {}
    users = np.asarray(users)
    for s in range(0, len(users), batch):
        ub = users[s:s + batch]
        UI = user_items_csr[ub]
        P = np.asarray(UI @ X)                                   
        P /= (np.linalg.norm(P, axis=1, keepdims=True) + 1e-9)
        S = P @ X.T                                              
        for j, u in enumerate(ub):
            items = UI.indices[UI.indptr[j]:UI.indptr[j + 1]]
            S[j, items] = -np.inf                                
            user_city = np.bincount(city_codes[items]).argmax()  
            S[j, city_codes != user_city] = -np.inf
            top = np.argpartition(-S[j], k)[:k]
            recs[int(u)] = top[np.argsort(-S[j][top])].tolist()
    return recs 

cb_recs = content_recs(users, trainval, k=K)
res_cb = evaluate(cb_recs, split['test_truth'], **final_args)
res_cb

Precision@5     0.001280
Recall@5        0.006400
MAP@5           0.003167
NDCG@5          0.003955
Coverage        0.109596
Novelty        15.796281
dtype: float64

Cold-Start խնդրի վերլուծությամբ ապացուցվեց, որ պատմություն չունեցող նոր օբյեկտների դեպքում կոլաբորատիվ մոդելները ձախողվում են, և միայն Content-Based մոդելն է կարողանում արդյունավետ աշխատել՝ հիմնվելով օբյեկտների հատկանիշների վրա։

In [6]:
from recsys_utils import popularity_recs

cold_truth = {u: it for u, it in split['test_truth'].items()
              if split['pop_trainval'][list(it)[0]] == 0}
print("Cold test examples:", len(cold_truth), "/", len(split['test_truth']))

pop_recs_final = popularity_recs(users, split['pop_trainval'], trainval, k=K)
knn_recs_final = implicit_recs(knn_model, users, trainval, k=K)
als_recs_final = implicit_recs(als_model, users, trainval, k=K)

cold_summary = pd.DataFrame({
    'Popularity':    evaluate(pop_recs_final, cold_truth, **final_args),
    'KNN':           evaluate(knn_recs_final, cold_truth, **final_args),
    'ALS':           evaluate(als_recs_final, cold_truth, **final_args),
    'Content-Based': evaluate(cb_recs,        cold_truth, **final_args),
}).T[METRIC_COLS]
cold_summary.index.name = 'Model'
display(cold_summary.round(5))
cold_summary.to_csv('../data/processed/cold_start_table.csv')

Cold test examples: 1359 / 10000


,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty
Model,,,,,,
Popularity,0.00000,0.00000,0.00000,0.00000,0.00002,10.84926
KNN,0.00000,0.00000,0.00000,0.00000,0.02220,16.23070
ALS,0.00000,0.00000,0.00000,0.00000,0.01330,12.71715
Content-Based,0.00088,0.00442,0.00227,0.00281,0.02247,16.00725


Այս բջիջները միավորում են բոլոր մոդելների արդյունքները մեկ ընդհանուր համեմատական աղյուսակում՝ ներառելով բոլոր հիմնական մետրիկները։

In [7]:
model_results.loc['Content-Based'] = res_cb[METRIC_COLS]

baseline_results = pd.read_pickle('../data/processed/results_baselines.pkl')
all_results = pd.concat([baseline_results, model_results])

long = all_results.reset_index().melt(id_vars='Model', var_name='Metric', value_name='Value')
summary = long.pivot_table(index='Model', columns='Metric', values='Value')
summary = summary.loc[['Random', 'Popularity', 'KNN', 'ALS', 'Content-Based'], METRIC_COLS]

display(summary.round(5))
summary.to_csv('../data/processed/summary_table.csv')

Metric,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty
Model,,,,,,
Random,0.00000,0.0000,0.00000,0.00000,0.16406,16.64623
Popularity,0.00010,0.0005,0.00021,0.00028,0.00002,10.84926
KNN,0.00098,0.0049,0.00275,0.00329,0.11776,16.26426
ALS,0.00260,0.0130,0.00727,0.00868,0.03313,12.59162
Content-Based,0.00128,0.0064,0.00317,0.00395,0.10960,15.79628


In [8]:
knn_tuning.to_csv('../data/processed/knn_tuning.csv', index=False)
als_tuning.to_csv('../data/processed/als_tuning.csv', index=False)

print("=== BEST KNN ===");  print(best_knn.to_dict())
print("=== BEST ALS ===");  print(best_als.to_dict())

print("\n=== SUMMARY ===")
print(summary.round(5).to_csv())

print("=== COLD START ===")
print(cold_summary.round(5).to_csv())

print("=== KNN TUNING ===")
print(knn_tuning.round(5).to_csv(index=False))

print("=== ALS TUNING ===")
print(als_tuning.round(5).to_csv(index=False))

=== BEST KNN ===
{'similarity': 'Cosine', 'neighbors': 50, 'Precision@5': 0.0009600000000000001, 'Recall@5': 0.0048, 'MAP@5': 0.00231, 'NDCG@5': 0.002916058387106106, 'Coverage': 0.09464268470201968, 'Novelty': 16.225664138793945, 'sec': 2.4}
=== BEST ALS ===
{'factors': 128.0, 'regularization': 0.01, 'alpha': 100.0, 'iterations': 15.0, 'Precision@5': 0.00244, 'Recall@5': 0.0122, 'MAP@5': 0.007026666666666666, 'NDCG@5': 0.00830467912483748, 'Coverage': 0.04172404050086016, 'Novelty': 13.294997215270996, 'sec': 22.3}

=== SUMMARY ===
Model,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty
Random,0.0,0.0,0.0,0.0,0.16406,16.64623
Popularity,0.0001,0.0005,0.00021,0.00028,2e-05,10.84926
KNN,0.00098,0.0049,0.00275,0.00329,0.11776,16.26426
ALS,0.0026,0.013,0.00727,0.00868,0.03313,12.59162
Content-Based,0.00128,0.0064,0.00317,0.00395,0.1096,15.79628

=== COLD START ===
Model,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty
Popularity,0.0,0.0,0.0,0.0,2e-05,10.84926
KNN,0.0,0.0,0.0,0.0,0.0222

In [9]:
import pandas as pd
reviews_df = pd.read_csv('../data/Reviews.csv', encoding='latin1')

print("Listings (clean):", listings_clean.shape[0], "| cities:", listings_clean['city'].nunique())
print(listings_clean['city'].value_counts().to_dict())
print("Reviews:", len(reviews_df), "| unique reviewers:", reviews_df['reviewer_id'].nunique())
print("Users (>=3 interactions):", split['n_users'], "| items:", split['n_items'])
print("Train nnz:", split['train'].nnz, "| trainval nnz:", split['trainval'].nnz)
print("Density:", split['trainval'].nnz / (split['n_users'] * split['n_items']))
print("Items with 0 interactions (trainval):", int((split['pop_trainval'] == 0).sum()))
print("Eval users:", len(split['eval_users']), "| test truth:", len(split['test_truth']))
print("Cold test examples:", len(cold_truth), "/", len(split['test_truth']))

print("\n=== ALS TUNING ===")
print(als_tuning[['factors','regularization','alpha','Recall@5','NDCG@5','Coverage','sec']].round(5).to_string(index=False))
print("\n=== KNN TUNING ===")
print(knn_tuning[['similarity','neighbors','Recall@5','NDCG@5','Coverage']].round(5).to_string(index=False))

Listings (clean): 279599 | cities: 10
{'Paris': 64628, 'New York': 36984, 'Sydney': 33630, 'Rome': 27641, 'Rio de Janeiro': 26608, 'Istanbul': 24517, 'Mexico City': 20058, 'Bangkok': 19360, 'Cape Town': 19086, 'Hong Kong': 7087}
Reviews: 5373143 | unique reviewers: 4450005
Users (>=3 interactions): 140786 | items: 279599
Train nnz: 248303 | trainval nnz: 389089
Density: 9.88448096023965e-06
Items with 0 interactions (trainval): 180992
Eval users: 10000 | test truth: 10000
Cold test examples: 1359 / 10000

=== ALS TUNING ===
 factors  regularization  alpha  Recall@5  NDCG@5  Coverage   sec
     128            0.01  100.0    0.0122 0.00830   0.04172  22.3
     128            0.10  100.0    0.0122 0.00830   0.04169  22.4
     256            0.01  100.0    0.0124 0.00822   0.05810 403.2
     256            0.10  100.0    0.0123 0.00821   0.05814 470.6
     256            0.10   40.0    0.0123 0.00818   0.04230 569.2
     256            0.01   40.0    0.0122 0.00814   0.04233 394.1
     128

Վիճակագրական ստուգմամբ համոզվել ենք, որ ALS մոդելն իրոք ավելի լավն է աշխատում, քան մյուսները։

In [ ]:
def per_user_metrics(recs, truth, k=5):
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    us = sorted(truth)
    hit, ndcg = np.zeros(len(us)), np.zeros(len(us))
    for n, u in enumerate(us):
        item = next(iter(truth[u]))
        rec = list(recs.get(u, []))[:k]
        if item in rec:
            hit[n] = 1.0
            ndcg[n] = disc[rec.index(item)]      
    return hit, ndcg

def bootstrap_ci(vec, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(vec), size=(n_boot, len(vec)))
    return np.percentile(vec[idx].mean(axis=1), [2.5, 97.5])

model_recs = {'Popularity': pop_recs_final, 'KNN': knn_recs_final,
              'ALS': als_recs_final, 'Content-Based': cb_recs}
per = {m: per_user_metrics(r, split['test_truth']) for m, r in model_recs.items()}

rows = []
for m, (hit, nd) in per.items():
    a, b = bootstrap_ci(hit); c, d = bootstrap_ci(nd)
    rows.append({'Model': m, 'Recall@5': round(hit.mean(), 5), 'Recall 95% CI': f'[{a:.4f}, {b:.4f}]',
                 'NDCG@5': round(nd.mean(), 5), 'NDCG 95% CI': f'[{c:.4f}, {d:.4f}]'})
ci_table = pd.DataFrame(rows).set_index('Model')
display(ci_table)
ci_table.to_csv('../data/processed/ci_table.csv')

print("\nPairwise Differences (NDCG@5):")
for a_, b_ in [('ALS', 'Content-Based'), ('Content-Based', 'KNN'), ('ALS', 'KNN')]:
    diff = per[a_][1] - per[b_][1]
    lo, hi = bootstrap_ci(diff)
    verdict = "Significant" if (lo > 0 or hi < 0) else "Not Significantly Different"
    print(f"{a_} − {b_}: {diff.mean():.5f}  95% CI [{lo:.5f}, {hi:.5f}] → {verdict}")

<>:31: SyntaxWarning: "\@" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\@"? A raw string is also an option.
<>:31: SyntaxWarning: "\@" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\@"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_23336\3883068790.py:31: SyntaxWarning: "\@" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\@"? A raw string is also an option.
  print("\nPairwise Differences (NDCG\@5):")


,Recall@5,Recall 95% CI,NDCG@5,NDCG 95% CI
Model,,,,
Popularity,0.0005,"[0.0001, 0.0010]",0.00028,"[0.0001, 0.0006]"
KNN,0.0049,"[0.0037, 0.0063]",0.00329,"[0.0024, 0.0043]"
ALS,0.0130,"[0.0108, 0.0152]",0.00868,"[0.0071, 0.0103]"
Content-Based,0.0064,"[0.0049, 0.0081]",0.00395,"[0.0030, 0.0050]"



Pairwise Differences (NDCG\@5):
ALS − Content-Based: 0.00472  95% CI [0.00291, 0.00652] → Significant
Content-Based − KNN: 0.00067  95% CI [-0.00068, 0.00195] → Not Significantly Different
ALS − KNN: 0.00539  95% CI [0.00374, 0.00709] → Significant


In [11]:
def content_recs_masked(users, user_items_csr, item_mask, k=5, batch=200):
    recs = {}
    users = np.asarray(users)
    for s in range(0, len(users), batch):
        ub = users[s:s + batch]
        UI = user_items_csr[ub]
        P = np.asarray(UI @ X)
        P /= (np.linalg.norm(P, axis=1, keepdims=True) + 1e-9)
        S = P @ X.T
        for j, u in enumerate(ub):
            items = UI.indices[UI.indptr[j]:UI.indptr[j + 1]]
            S[j, items] = -np.inf
            user_city = np.bincount(city_codes[items]).argmax()
            S[j, city_codes != user_city] = -np.inf
            S[j, ~item_mask] = -np.inf
            top = np.argpartition(-S[j], k)[:k]
            recs[int(u)] = top[np.argsort(-S[j][top])].tolist()
    return recs

def hybrid_recs(als_recs, cb_cold_recs, n_cold, k=5):
    out = {}
    for u in als_recs:
        rec = als_recs[u][:k - n_cold]
        for i in cb_cold_recs[u][:n_cold]:
            if i not in rec:
                rec.append(i)
        for i in als_recs[u]:                  
            if len(rec) >= k: break
            if i not in rec: rec.append(i)
        out[u] = rec[:k]
    return out

cold_mask = split['pop_trainval'] == 0
cb_cold = content_recs_masked(users, trainval, cold_mask, k=K)

rows = []
for n_cold in [0, 1, 2, 3]:
    h = hybrid_recs(als_recs_final, cb_cold, n_cold, K)
    a = evaluate(h, split['test_truth'], **final_args)
    c = evaluate(h, cold_truth, **final_args)
    rows.append({'cold_slots': n_cold, 'Recall@5': a['Recall@5'], 'NDCG@5': a['NDCG@5'],
                 'Coverage': a['Coverage'], 'Cold Recall@5': c['Recall@5'], 'Cold NDCG@5': c['NDCG@5']})
hybrid_table = pd.DataFrame(rows).set_index('cold_slots')
display(hybrid_table.round(5))
hybrid_table.to_csv('../data/processed/hybrid_table.csv')
print(hybrid_table.round(5).to_csv())

,Recall@5,NDCG@5,Coverage,Cold Recall@5,Cold NDCG@5
cold_slots,,,,,
0,0.0130,0.00868,0.03313,0.00000,0.00000
1,0.0116,0.00814,0.05328,0.00442,0.00171
2,0.0098,0.00738,0.06481,0.00589,0.00247
3,0.0083,0.00666,0.07213,0.00736,0.00341


cold_slots,Recall@5,NDCG@5,Coverage,Cold Recall@5,Cold NDCG@5
0,0.013,0.00868,0.03313,0.0,0.0
1,0.0116,0.00814,0.05328,0.00442,0.00171
2,0.0098,0.00738,0.06481,0.00589,0.00247
3,0.0083,0.00666,0.07213,0.00736,0.00341



In [ ]:
import pickle, numpy as np, pandas as pd
from recsys_utils import evaluate, city_popularity_recs, METRIC_COLS

split = pickle.load(open('../data/processed/split.pkl', 'rb'))
listings_clean = pd.read_pickle('../data/processed/listings_clean.pkl')
city_codes = listings_clean['city'].astype('category').cat.codes.values.astype(int) + 1
trainval, users = split['trainval'], split['eval_users']
final_args = dict(item_pop=split['pop_trainval'], n_users=split['n_users'],
                  n_items=split['n_items'], k=5)

recs = city_popularity_recs(users, split['pop_trainval'], city_codes, trainval, k=5)
res = evaluate(recs, split['test_truth'], **final_args)

summary = pd.read_csv('../data/processed/summary_table.csv', index_col='Model')
summary = summary.drop(index='Popularity (city-aware)', errors='ignore')
summary.loc['Popularity (city-aware)'] = res[METRIC_COLS]
summary = summary.loc[['Random', 'Popularity', 'Popularity (city-aware)', 'KNN', 'ALS', 'Content-Based']]
summary.to_csv('../data/processed/summary_table.csv')
display(summary.round(5))